# Building a Reproducible Mental Health Data Ecosystem: The Kilifi County, Kenya FAIR² Model Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the "Building a Reproducible Mental Health Data Ecosystem: The Kilifi County, Kenya FAIR² Model" dataset using the `mlcroissant` library, following reproducible best practices.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Metadata is mlcroissant.Metadata, use .to_json() to access as dict
metadata = dataset.metadata.to_json()

print("Dataset Title:", metadata.get('name'))
print("Description:", metadata.get('description'))

# Show dataset's Croissant identifier
print("Identifier (@id):", metadata.get('@id'))
print("Version:", metadata.get('version'))

## 2. Data Overview
Review available record sets, fields, and their IDs. We list the record sets, fields and columns in the dataset. All entities are referenced by their `@id`.

In [ ]:
# Fetch all record sets
record_sets = dataset.metadata.record_sets

print("Record Sets (@id and name):")
for rs in record_sets:
    print(f"  @id: {rs['@id']} | name: {rs.get('name', '(unnamed)')}")

# For each record set, print its available fields and columns (by @id)
print("\nFields and columns per record set:")
for rs in record_sets:
    print(f"\nRecord set: {rs['@id']} ({rs.get('name', '(unnamed)')})")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    if not fields:
        print("  No fields defined.")
        continue
    for field in fields:
        if isinstance(field, str):
            print(f"    Field @id: {field}")
        elif isinstance(field, dict):
            print(f"    Field @id: {field.get('@id')} | name: {field.get('name', '')}")
    columns = rs.get('column', [])
    if isinstance(columns, dict):
        columns = [columns]
    if columns:
        print("  Columns:")
        for column in columns:
            if isinstance(column, str):
                print(f"    Column @id: {column}")
            elif isinstance(column, dict):
                print(f"    Column @id: {column.get('@id')} | name: {column.get('name', '')}")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# List all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Load all records as dicts
    recs = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(recs)
    dataframes[record_set_id] = df
    print(f"Loaded Record Set '@id': {record_set_id}, {df.shape[0]} rows, Columns: {list(df.columns)[:8]}{'...' if len(df.columns) > 8 else ''}")

# As an example, display the first few rows of the first record set
example_rs_id = record_set_ids[0] if record_set_ids else None
if example_rs_id:
    print(f"\nFirst 5 rows from record set '@id': {example_rs_id}")
    display(dataframes[example_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes to prepare your dataset for further analysis.

For this example, we will:
- Select the record set containing the main survey data (`@id` as identified above)
- Choose a numeric field (e.g., the GAD-7 score, PHQ-9 score, etc. – referenced by its @id as displayed in the overview)
- Perform basic filtering and normalization
- Optionally group by a demographic attribute (such as gender or education level, referenced by @id)

**Modify the variables below to adjust which fields are selected!**

In [ ]:
# --- Specify these according to your data schema overview ---
# Example field @ids (replace by actual @ids from overview step)

# Example: Use the first record set,
record_set_id = record_set_ids[0] if record_set_ids else None
# Example: The GAD-7 (Generalized Anxiety Disorder) assessment total score may be something like 'gad7_total@id'
numeric_field_id = None
group_field_id = None

# Try to guess a numeric field, fallback if not found
if record_set_id:
    df = dataframes[record_set_id]
    for col in df.columns:
        if 'gad' in col.lower() or 'phq' in col.lower() or 'score' in col.lower():
            numeric_field_id = col
            break
    # Try guessing a grouping field
    for col in df.columns:
        if any(x in col.lower() for x in ['gender','education','village','age_group','group']):
            group_field_id = col
            break

# If no such fields, print available columns
if numeric_field_id is None or group_field_id is None:
    print("Available columns:", list(df.columns))

# Make sure the field exists
if numeric_field_id and numeric_field_id in df.columns:
    # Drop NA values in the field
    filtered_df = df[df[numeric_field_id].notnull()]
    print(f"\nDistribution of {numeric_field_id} (before filtering):")
    print(filtered_df[numeric_field_id].describe())

    # Remove outliers (thresholding at mean+3std)
    upper = filtered_df[numeric_field_id].mean() + 3 * filtered_df[numeric_field_id].std()
    filtered_df = filtered_df[filtered_df[numeric_field_id] <= upper]
    print(f"Filtered to exclude values above {upper:.2f}")
    
    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # If grouping variable available, group and show means
    if group_field_id and group_field_id in filtered_df.columns:
        print(f"\nAverages by group {group_field_id}:")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(grouped_df.head())
else:
    print("No suitable numeric field detected – please update 'numeric_field_id' above.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For instance, we can plot the histogram of the selected numeric field, and the mean score per group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_field_id and numeric_field_id in dataframes[record_set_id].columns:
    df = dataframes[record_set_id]
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Histogram of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(9,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
In this notebook, we've loaded the Kilifi County, Kenya FAIR² mental health dataset using `mlcroissant`, explored its record sets and fields, performed basic data cleaning and normalization, and visualized important variables.

This FAIR dataset is ready for advanced analytic workflows such as regression, classification, or public health reporting. Adjust field selections according to your analytic goals.

**References:**
- [FAIR² dataset Croissant schema](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)
- [mlcroissant documentation](https://github.com/mlcommons/croissant)

Remember to always reference all entities (record sets, fields, columns) by their `@id` as discovered in step 2, for durable code that works with Croissant schemas.